# arcedge Stage 2 — GPU batched shortest paths (Colab)

Builds arcedge with the CUDA backend and benchmarks the Lagrangian solver's
shortest-path subproblem on a real San Francisco time-expanded instance
(~1.7M arcs, 150 commodities) across three backends:

- `dijkstra` — per-commodity binary-heap Dijkstra, CPU threads (Stage 1 baseline)
- `dag` — topological level sweep, CPU threads (the GPU kernel's exact structure, FP64)
- `cuda` — the same level sweep on GPU: one bulk-relaxation kernel per time layer,
  all commodities batched, graph resident on device across subgradient iterations, FP32

**Runtime → Change runtime type → GPU (T4 works; A100 better).**

In [ ]:
!nvidia-smi

## Get the code

The repo is private: either make a fine-grained GitHub token and paste it below,
or upload a zip of the repo to `/content/` and unzip it instead.

In [ ]:
import os
BRANCH = 'claude/stage-1-implementation-plan-9lv5f3'  # or 'main' once merged
if not os.path.exists('arcedge'):
    token = ''  # <-- paste a GitHub token here if the repo is private
    url = f'https://{token}@github.com/fhk/arcedge.git' if token else 'https://github.com/fhk/arcedge.git'
    !git clone --branch {BRANCH} {url} arcedge
%cd /content/arcedge
!git log --oneline -3

In [ ]:
!pip install -q highspy numpy pyarrow shapely
!cmake -B build -DCMAKE_BUILD_TYPE=Release -DARCEDGE_CUDA=ON 2>&1 | tail -2
!cmake --build build -j2 2>&1 | tail -3

In [ ]:
# Unit tests: correctness identities incl. DAG-vs-Dijkstra equivalence
!ctest --test-dir build --output-on-failure

## Benchmark: SF time-expanded MCF, three backends

Generates the 1,683,885-arc instance from the committed SF street graph, then
solves it with each backend. `dijkstra`/`dag` are FP64 on CPU; `cuda` is FP32
on GPU, so its bounds may differ in the 5th–6th significant digit — the gap
must land at ~0.68% for all three.

In [ ]:
!mkdir -p data
!./build/arcedge gen --street data/sf_streets.graph --out data/sf_te.txt \
    --time 36 --commodities 150 --cap 8 --hubs 5 --hub-frac 0.5 --seed 17
import subprocess, re, json
results = {}
for backend in ['dijkstra', 'dag', 'cuda']:
    out = subprocess.run(
        ['./build/arcedge', 'solve', 'data/sf_te.txt', '--iters', '120',
         '--tol', '0.01', '--primal-every', '10', '--sp-backend', backend,
         '--quiet'],
        capture_output=True, text=True)
    line = out.stdout.strip().splitlines()[-1]
    print(f'{backend:9s} {line}')
    m = re.search(r'lb ([\d.]+)\s+ub ([\d.]+)\s+gap ([\d.]+)%\s+iters (\d+)\s+time ([\d.]+) ms', line)
    results[backend] = dict(lb=float(m[1]), ub=float(m[2]), gap=float(m[3]),
                            iters=int(m[4]), ms=float(m[5]))
print(json.dumps(results, indent=2))

In [ ]:
# Equivalence + speedup report. CUDA is FP32: allow 1e-4 relative slack on
# the bounds; the CPU backends must agree to near machine precision.
ref = results['dijkstra']
assert abs(results['dag']['lb'] - ref['lb']) <= 1e-9 * ref['lb'], 'dag LB mismatch'
assert abs(results['cuda']['lb'] - ref['lb']) <= 1e-4 * ref['lb'], 'cuda LB mismatch (beyond FP32 tolerance)'
assert abs(results['cuda']['ub'] - ref['ub']) <= 1e-4 * ref['ub'], 'cuda UB mismatch (beyond FP32 tolerance)'
print('EQUIVALENCE OK')
for b in ['dag', 'cuda']:
    print(f"{b}: {ref['ms'] / results[b]['ms']:.2f}x vs dijkstra "
          f"({results[b]['ms']:.0f} ms vs {ref['ms']:.0f} ms)")

## Big-batch SSSP throughput: K = 1000 commodities

The 150-commodity batch barely loads a GPU — the batch dimension is where it
scales. This benchmark isolates the shortest-path subproblem: `--no-primal`
disables the (sequential, CPU) primal heuristic and the FP64 certification
pass, so wall-clock ≈ iterations × batched-SSSP. CPU backends scale with
cores; the GPU relaxes all 1000 commodities per kernel launch.

GPU memory for the packed distance/parent matrix: 1000 × 413,532 nodes × 8 B
≈ 3.3 GB (fits a T4's 16 GB).

In [ ]:
!./build/arcedge gen --street data/sf_streets.graph --out data/sf_te_k1000.txt \
    --time 36 --commodities 1000 --cap 20 --hubs 8 --hub-frac 0.5 --seed 17
import subprocess, re
ITERS = 20
bench = {}
for backend in ['dag', 'cuda']:
    out = subprocess.run(
        ['./build/arcedge', 'solve', 'data/sf_te_k1000.txt', '--iters', str(ITERS),
         '--no-primal', '--sp-backend', backend, '--quiet'],
        capture_output=True, text=True)
    line = out.stdout.strip().splitlines()[-1]
    print(f'{backend:5s} {line}')
    bench[backend] = float(re.search(r'time ([\d.]+) ms', line)[1])
print(f'\nper-iteration over {ITERS} iters (primal + certification disabled):')
for b, ms in bench.items():
    print(f'  {b:5s} {ms / ITERS:8.1f} ms/iter')
print(f"  cuda speedup vs dag (CPU): {bench['dag'] / bench['cuda']:.2f}x")

## Optional: full E2E suite (synthetic + real data, HiGHS validations)

~5 minutes on a Colab CPU; exercises everything in `PLAN.md`'s proof table.

In [ ]:
!bash scripts/run_e2e.sh

## Notes

- Paths never leave the GPU: per-arc loads are accumulated device-side and
  only ~6.7 MB (loads + K distances) crosses PCIe per iteration, not the
  K×N packed matrix (~0.5 GB on this instance).
- The final lower bound is **FP64-certified**: L(λ) is re-evaluated on CPU
  at the best multipliers (any λ ≥ 0 yields a valid bound), so FP32 rounding
  cannot produce an invalid bound. Run without `--quiet` to see the
  `fp64 certification:` line with both values.
- The `solve` wall-clock still includes the sequential CPU primal heuristic —
  it becomes the Amdahl limit once SSSP is fast; LNS/GPU primal work is
  S2-M2+. For a pure SSSP throughput comparison, raise `--commodities`
  (e.g. 1000) when generating the instance: the GPU batch scales, the CPU
  backends scale only with cores.
- Stage 2 plan and milestones: see `PLAN.md`.